Hace tiempo repliqué la funcionalidad del comando OUTLIERS de ACL en python. Este comando es utilizado en auditoría para encontrar valores atípicos, es decir valores que están fuera del promedio de actividad de un cliente, utiliza la desviación estándar para saber que tan alejados están los valores del promedio. Es un muy buen enfoque, pero investigando un poco encontré que:

1. La desviación estándar es sensible a los propios valores atípicos, ejemplo hay un cliente C0001 que tiene operaciones cuyos importes son 1_000, 2_000, 1,500. Su promedio es de 1,500. Tiene una última operación de 5_000 que por sí misma es un valor atípico (superior al promedio) y debería de ser investigado, pero al considerar todas las operaciones del cliente en su conjunto (1,000 + 2,000 + 1,500 + 5,000) / 4 = 2,375; la media sube considerablemente, por lo cual hay riesgo de que la operación "se termine ocultando".
2. Se necesitan grupos grandes, es decir, necesitamos que los clientes tengan varias operaciones, preferentemente más de 20.
3. El comportamiento del cliente debe tener cierta simetría, cercana a un comportamiento gaussiano (forma de campana), donde graficamente en el centro se agruparán la mayoría de las operaciones y en sus extremos estarán los valores atípicos o valores extremos.

In [ ]:
import pandas as pd

def outliers(df, columna_analisis, columna_a_agrupar, multiplo_std, estadistico):
    """
    Identifica valores atípicos sobre un dataframe agrupando por una columna específica y calculando la media o mediana y la desviación estándar de la columna bajo análisis.
    df = dataframe de entrada
    columna_analisis = nombre de la columna sobre la cual se realizará el análisis (debe ser cuantificable)
    columna_a_agrupar = nombre de la columna por la cual se realizará el análisis (debe ser categórico)
    multiplo_std = múltiplo de la desviación estándar para definir los límites: 1, 2, 3
    estadistico = 'mean' o 'median' para definir si se calculará la media o la mediana
    """

    # verificar que estadistico sea 'mean' o 'median'
    if estadistico not in ['mean', 'median']:
        raise ValueError("El parámetro 'estadistico' debe ser 'mean' o 'median'.")

    # cálculo de la media y la desviación estándar por número de cliente
    grupo_clientes = df.groupby(columna_a_agrupar)[columna_analisis].agg([estadistico, 'std'])
    grupo_clientes.rename(columns={estadistico: 'Media', 'std': 'Desviación_Estándar'}, inplace=True)
    grupo_clientes['Límite_Inferior'] = grupo_clientes['Media'] - (multiplo_std * grupo_clientes['Desviación_Estándar'])
    grupo_clientes['Límite_Superior'] = grupo_clientes['Media'] + (multiplo_std * grupo_clientes['Desviación_Estándar'])
    grupo_clientes.reset_index(inplace=True)

    # identificación de valores atípicos utilizando los límites específicos de cada cliente
    df = df.merge(grupo_clientes[[columna_a_agrupar, 'Límite_Inferior', 'Límite_Superior']], on=columna_a_agrupar, how='left')
    df['Status'] = ""
    df.loc[df[columna_analisis] > df['Límite_Superior'], 'Status'] = "Superior"
    df.loc[df[columna_analisis] < df['Límite_Inferior'], 'Status'] = "Inferior"

    # filtrado de valores atípicos
    valores_atipicos = df[df['Status'] != ""]
    valores_atipicos = valores_atipicos.sort_values(by=columna_a_agrupar)    
    return valores_atipicos

In [2]:
# importar datos, cuyo origen se obtienen de la base de datos muestra de ACL
df = pd.read_excel(r'data\Ar_.xls')

In [3]:
# se ingresan en la función los parámetros requridos, son iguales a los obtenidos por IDEA
outliers(df,'Trans_Amount','Cust_Number',3,'median')

,Cust_Number,Invoice_Date,Due_Date,Ref_No,Trans_Type,Trans_Amount,Límite_Inferior,Límite_Superior,Status
98,65003,2000-09-22,2000-03-21,213511,IN,4954.64,-2396.39,3723.75,Superior
30,262001,2000-09-02,2000-03-01,213295,IN,3567.34,-1878.22,2779.56,Superior
696,262001,2000-12-04,2000-11-15,213840,PM,-2044.82,-1878.22,2779.56,Inferior
732,376005,2000-12-04,2000-10-03,213309,PM,-931.55,-737.49,1771.80,Inferior
56,501657,2000-09-09,2000-03-08,213390,IN,5549.19,-3890.61,4184.21,Superior
41,501657,2000-09-03,2000-03-02,213325,IN,4426.14,-3890.61,4184.21,Superior
469,811002,2000-11-19,2000-12-19,214155,IN,3409.82,-1295.61,2544.67,Superior
54,925007,2000-09-08,2000-03-07,213357,IN,2972.78,-1503.54,2939.30,Superior
425,925007,2000-11-13,2000-12-13,214095,IN,3393.87,-1503.54,2939.30,Superior
186,925007,2000-10-08,2000-01-06,213771,IN,3030.71,-1503.54,2939.30,Superior


¿Cual es el valor agregado de este notebook?, quiero introducir la idea del análisis intercuartílico (interquartile range) IQR por sus siglas en inglés, el cual:
1. se basa en percentiles 25%, 50%, 75% de los datos, por lo que los valores atípicos o extremos influyen poco
2. funciona en distribuciones no normales o sesgadas

La propuesta:
Si sabemos que hay muchas operaciones por clientes >20-30, elegiría IQR por ser más robusto y resiste mejor observaciones erróneas o extraordinarias.

In [ ]:
# IQR funciona sobre clientes por más número de operaciones, por tanto en este ejemplo se toman los clientes con 10 o más
# pero este debe evaluarse detenidamente, no deberíamos dejar fuera ninguna transacción   
df_filtered = df.groupby('Cust_Number').filter(lambda x: len(x) >= 10)
df_filtered.head()

,Cust_Number,Invoice_Date,Due_Date,Ref_No,Trans_Type,Trans_Amount
3,516372,2000-02-17,2000-03-18,211206,IN,1610.87
4,516372,2000-04-30,2000-03-18,211206,TR,-1298.43
5,518008,2000-05-21,2000-06-20,212334,CN,-12.23
7,518008,2000-06-10,2000-07-10,212592,CN,-37.15
8,501657,2000-06-30,2000-07-30,212824,IN,1524.32


In [5]:
# propuesta, añadiendo Q1, Q3 e IQR a cada fila
df_filtered['Q1'] = (
    df_filtered.groupby('Cust_Number')['Trans_Amount']
      .transform(lambda x: x.quantile(0.25))
)

df_filtered['Q3'] = (
    df_filtered.groupby('Cust_Number')['Trans_Amount']
      .transform(lambda x: x.quantile(0.75))
)

df_filtered['IQR'] = df_filtered['Q3'] - df_filtered['Q1']
df_filtered

,Cust_Number,Invoice_Date,Due_Date,Ref_No,Trans_Type,Trans_Amount,Q1,Q3,IQR
3,516372,2000-02-17,2000-03-18,211206,IN,1610.87,-19.23,841.50,860.73
4,516372,2000-04-30,2000-03-18,211206,TR,-1298.43,-19.23,841.50,860.73
5,518008,2000-05-21,2000-06-20,212334,CN,-12.23,340.45,620.88,280.43
7,518008,2000-06-10,2000-07-10,212592,CN,-37.15,340.45,620.88,280.43
8,501657,2000-06-30,2000-07-30,212824,IN,1524.32,-96.62,929.97,1026.59
...,...,...,...,...,...,...,...,...,...
766,925007,2000-12-10,2000-01-09,214378,IN,111.96,517.85,1189.11,671.26
767,925007,2000-12-10,2000-01-09,214386,IN,808.20,517.85,1189.11,671.26
768,65003,2000-12-10,2000-01-09,214374,IN,73.40,136.49,1321.24,1184.75
769,641464,2000-12-10,2000-01-09,214392,IN,46.77,399.63,1203.92,804.29


In [6]:
# crear función de detección de outliers basada en IQR
def detectar_outliers(grupo):
    Q1 = grupo['Trans_Amount'].quantile(0.25)
    Q3 = grupo['Trans_Amount'].quantile(0.75)

    IQR = Q3 - Q1

    lim_inf = Q1 - 3 * IQR
    lim_sup = Q3 + 3 * IQR

    grupo['is_outlier'] = (
        (grupo['Trans_Amount'] < lim_inf) |
        (grupo['Trans_Amount'] > lim_sup)
    )

    return grupo

In [7]:
# identificar valores atípicos
outliers = (
    df_filtered
    .groupby('Cust_Number')
    .apply(detectar_outliers)
    .reset_index()
    .drop(columns='level_1')
)

outliers[outliers.is_outlier == True]

,Cust_Number,Invoice_Date,Due_Date,Ref_No,Trans_Type,Trans_Amount,Q1,Q3,IQR,is_outlier
17,65003,2000-09-22,2000-03-21,213511,IN,4954.64,136.49,1321.24,1184.75,True
141,222006,2000-11-19,2000-12-19,214141,IN,2487.30,342.11,776.80,434.69,True
146,262001,2000-09-02,2000-03-01,213295,IN,3567.34,32.09,865.10,833.01,True
244,297397,2000-09-15,2000-03-14,213476,IN,3303.09,235.39,891.50,656.11,True
248,297397,2000-11-04,2000-05-03,214026,IN,3038.84,235.39,891.50,656.11,True
288,376005,2000-12-04,2000-10-03,213309,PM,-931.55,379.09,675.88,296.79,True
296,501657,2000-09-03,2000-03-02,213325,IN,4426.14,-96.62,929.97,1026.59,True
298,501657,2000-09-09,2000-03-08,213390,IN,5549.19,-96.62,929.97,1026.59,True
541,812465,2000-08-21,2000-11-29,213227,IN,3582.98,272.00,1067.25,795.25,True
550,812465,2000-12-04,2000-11-29,213227,PM,-3582.98,272.00,1067.25,795.25,True


### Conclusión

Si comparas los resultados de la función creada "outliers" de python, la cual arroja los mismos resultados que ACL contra los resultados del análisis IQR son muy similares, en conclusión y siendo conservadores, utilizaría una combinación de ambos para realizar una solicitud de pólizas o información física.   